# Databricks AI Functions

**Dataset:** `samples.bakehouse.media_customer_reviews`, `samples.bakehouse.sales_franchises`

**Difficulty:** Medium

**Topics:** `ai_analyze_sentiment`, `ai_classify`, `ai_summarize`, SQL AI functions via `spark.sql`, combining AI results with aggregations

> **Databricks-only:** These functions call foundation models hosted on Databricks.
> They are available in Databricks Runtime 13.2+ on clusters or SQL warehouses with **Unity Catalog enabled**.
> They are **not available** in open-source Spark or local environments.
>
> Each function call invokes a model endpoint - apply them to filtered/sampled data rather than full large tables to manage cost and latency.

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

df_reviews    = spark.table("samples.bakehouse.media_customer_reviews")
df_franchises = spark.table("samples.bakehouse.sales_franchises")

# Register views for SQL-based AI functions
df_reviews.createOrReplaceTempView("reviews")
df_franchises.createOrReplaceTempView("franchises")

## Problem 1 - Sentiment Analysis on Reviews

Use the `ai_analyze_sentiment` SQL function to classify each customer review as `positive`, `negative`, or `neutral`.

Apply it to a sample of 100 reviews to keep runtime short.
Join with `franchises` to enrich the result with the franchise name and country.

**Expected output columns:** `new_id`, `franchiseID`, `franchise_name`, `country`, `review`, `sentiment`

> `ai_analyze_sentiment` is a SQL function - use it via `spark.sql()` or `%sql` magic.
> It accepts a string column and returns one of: `'positive'`, `'negative'`, `'neutral'`, `'mixed'`.

In [ ]:
# Problem 1 - write your solution here
# Assign result to: result_1

result_1 = None  # replace this

In [ ]:
# ── Tests for Problem 1 ──────────────────────────────────────────
assert result_1 is not None, "result_1 is None"
assert hasattr(result_1, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_1.columns]
assert 'sentiment' in cols, "Missing column: sentiment"
assert 'review' in cols, "Missing column: review"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_1.count()
assert cnt > 0, "Got 0 rows"
valid_sentiments = {'positive', 'negative', 'neutral', 'mixed'}
actual_sentiments = {r['sentiment'].lower() for r in result_1.select('sentiment').distinct().collect() if r['sentiment']}
assert actual_sentiments.issubset(valid_sentiments), f"Unexpected sentiment values: {actual_sentiments - valid_sentiments}"
print(f"Problem 1 passed ✓  ({cnt} reviews analysed, sentiments: {actual_sentiments})")

## Problem 2 - Sentiment Distribution per Franchise

Building on Problem 1 (or re-running the sentiment analysis), aggregate the results:
- Count reviews per franchise per sentiment value
- Compute `percentageOfReviews` = count for that sentiment / total reviews for that franchise × 100

Sort by `franchiseID` and `sentiment`.

**Expected output columns:** `franchiseID`, `franchise_name`, `country`, `sentiment`, `numberReviews`, `percentageOfReviews`

In [ ]:
# Problem 2 - write your solution here
# Assign result to: result_2

result_2 = None  # replace this

In [ ]:
# ── Tests for Problem 2 ──────────────────────────────────────────
assert result_2 is not None, "result_2 is None"
assert hasattr(result_2, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_2.columns]
assert 'sentiment' in cols, "Missing column: sentiment"
assert 'numberreviews' in cols or 'numberreviews' in cols or 'number_reviews' in cols or 'numberreviews' in [c.replace(' ','') for c in cols], "Missing column: numberReviews"
assert 'percentageofreviews' in cols or 'percentage_of_reviews' in [c.lower().replace(' ','_') for c in result_2.columns], "Missing column: percentageOfReviews"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_2.count()
assert cnt > 0, "Got 0 rows"
# Percentages should be between 0 and 100
pct_col = [c for c in result_2.columns if 'percent' in c.lower() or 'pct' in c.lower()][0]
max_pct = result_2.agg(F.max(pct_col)).collect()[0][0]
assert max_pct <= 100.0, f"percentageOfReviews exceeds 100: {max_pct}"
print(f"Problem 2 passed ✓  ({cnt} franchise-sentiment combinations)")

## Problem 3 - Which Franchise Has the Most Positive Reviews?

From the sentiment analysis results, find the franchise with the highest percentage of `positive` reviews
(minimum 5 reviews to qualify). Show the top 5 franchises by positive review percentage.

**Expected output columns:** `franchiseID`, `franchise_name`, `country`, `positive_count`, `total_reviews`, `positive_pct`

In [ ]:
# Problem 3 - write your solution here
# Assign result to: result_3

result_3 = None  # replace this

In [ ]:
# ── Tests for Problem 3 ──────────────────────────────────────────
assert result_3 is not None, "result_3 is None"
assert hasattr(result_3, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_3.columns]
assert 'positive_pct' in cols or any('positive' in c and 'pct' in c for c in cols), "Missing: positive_pct"
assert 'total_reviews' in cols, "Missing column: total_reviews"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_3.count()
assert 0 < cnt <= 5, f"Expected 1-5 rows (top 5), got {cnt}"
total_col = [c for c in result_3.columns if 'total' in c.lower()][0]
min_reviews = result_3.agg(F.min(total_col)).collect()[0][0]
assert min_reviews >= 5, f"All franchises should have >= 5 reviews, min was {min_reviews}"
print(f"Problem 3 passed ✓  ({cnt} top franchises)")

## Problem 4 - Classify Review Topics

Use `ai_classify` to classify each review into one of these categories:
`'product quality'`, `'customer service'`, `'price'`, `'ambience'`, `'other'`.

Apply it to a sample of 50 reviews.

**Expected output columns:** `new_id`, `review`, `topic`, `sentiment`

> `ai_classify(column, ARRAY('cat1','cat2',...))` assigns the most relevant label from the provided list.
> Combine with `ai_analyze_sentiment` in the same query for dual classification.

In [ ]:
# Problem 4 - write your solution here
# Assign result to: result_4

result_4 = None  # replace this

In [ ]:
# ── Tests for Problem 4 ──────────────────────────────────────────
assert result_4 is not None, "result_4 is None"
assert hasattr(result_4, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_4.columns]
assert 'topic' in cols, "Missing column: topic"
assert 'review' in cols, "Missing column: review"
assert len(cols) == 2, f"Expected exactly 2 columns, got {len(cols)}: {cols}"
cnt = result_4.count()
assert cnt > 0, "Got 0 rows"
valid_topics = {'product quality', 'customer service', 'price', 'ambience', 'other'}
actual_topics = {r['topic'].lower() for r in result_4.select('topic').distinct().collect() if r['topic']}
assert actual_topics.issubset(valid_topics), f"Unexpected topics: {actual_topics - valid_topics}"
print(f"Problem 4 passed ✓  ({cnt} reviews classified, topics: {actual_topics})")

## Problem 5 - Topic & Sentiment Heatmap Data

Combine `ai_analyze_sentiment` and `ai_classify` on a sample of reviews.
Produce a summary table showing the count of reviews per `topic` × `sentiment` combination.
This is the data that would power a heatmap visualisation.

**Expected output columns:** `topic`, `sentiment`, `review_count`

In [ ]:
# Problem 5 - write your solution here
# Assign result to: result_5

result_5 = None  # replace this

In [ ]:
# ── Tests for Problem 5 ──────────────────────────────────────────
assert result_5 is not None, "result_5 is None"
assert hasattr(result_5, 'columns'), "Must be a Spark DataFrame"
cols = [c.lower() for c in result_5.columns]
assert 'topic' in cols, "Missing column: topic"
assert 'sentiment' in cols, "Missing column: sentiment"
assert 'review_count' in cols, "Missing column: review_count"
assert len(cols) == 3, f"Expected exactly 3 columns, got {len(cols)}: {cols}"
cnt = result_5.count()
assert cnt > 0, "Got 0 rows"
min_count = result_5.agg(F.min('review_count')).collect()[0][0]
assert min_count >= 1, "review_count should be >= 1"
print(f"Problem 5 passed ✓  ({cnt} topic-sentiment combinations)")